In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp04/ex_5.py ──
"""
Shared infrastructure for MLFP04 Exercise 5 — Association Rules.

Contains: synthetic Singapore retail transaction generator, category map,
rule dataclass helpers, and small polars utilities used by every technique
file in ``modules/mlfp04/solutions/ex_5/``.

Technique-specific code (Apriori from scratch, FP-Growth wrapper, rule
evaluation, feature engineering for classification) does NOT belong here —
it lives in the per-technique files.
"""

import asyncio
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import polars as pl

from kailash_ml import ExperimentTracker


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp04_ex5_association_rules"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# SINGAPORE RETAIL PRODUCT CATALOGUE
# ════════════════════════════════════════════════════════════════════════
# 25 products grouped to mirror a typical HDB neighbourhood mini-mart basket.

PRODUCTS: list[str] = [
    "bread",
    "butter",
    "milk",
    "eggs",
    "rice",
    "noodles",
    "soy_sauce",
    "cooking_oil",
    "chicken",
    "fish",
    "coffee",
    "tea",
    "sugar",
    "condensed_milk",
    "biscuits",
    "chips",
    "soft_drink",
    "beer",
    "wine",
    "tissue",
    "shampoo",
    "soap",
    "detergent",
    "toothpaste",
    "bananas",
]

CATEGORY_MAP: dict[str, str] = {
    "bread": "breakfast",
    "butter": "breakfast",
    "eggs": "breakfast",
    "milk": "dairy",
    "condensed_milk": "dairy",
    "coffee": "beverage",
    "tea": "beverage",
    "soft_drink": "beverage",
    "sugar": "pantry",
    "rice": "pantry",
    "cooking_oil": "pantry",
    "soy_sauce": "pantry",
    "noodles": "pantry",
    "chicken": "protein",
    "fish": "protein",
    "beer": "alcohol",
    "wine": "alcohol",
    "chips": "snack",
    "biscuits": "snack",
    "bananas": "fruit",
    "shampoo": "personal_care",
    "soap": "personal_care",
    "toothpaste": "personal_care",
    "tissue": "household",
    "detergent": "household",
}

# Co-purchase bundles (items, probability). Models real behaviour:
# kaya-toast breakfast, kopi-C, household replenishment, beer+chips, etc.
BUNDLES: list[tuple[list[str], float]] = [
    (["bread", "butter", "eggs"], 0.15),
    (["coffee", "condensed_milk", "sugar"], 0.12),
    (["rice", "chicken", "soy_sauce"], 0.10),
    (["noodles", "eggs", "soy_sauce"], 0.08),
    (["beer", "chips"], 0.09),
    (["milk", "biscuits"], 0.07),
    (["shampoo", "soap", "toothpaste"], 0.06),
    (["tea", "sugar", "biscuits"], 0.05),
    (["wine", "chips", "biscuits"], 0.04),
    (["cooking_oil", "rice", "fish"], 0.06),
    (["detergent", "tissue", "soap"], 0.05),
    (["bananas", "milk", "eggs"], 0.05),
]

N_TRANSACTIONS_DEFAULT: int = 2500


# ════════════════════════════════════════════════════════════════════════
# TRANSACTION GENERATOR
# ════════════════════════════════════════════════════════════════════════


def _draw_basket(rng: np.random.Generator, bundle_probs: Iterable[float]) -> set[str]:
    """Draw one basket: each bundle fires with its probability, each item in
    a firing bundle is kept with 0.85 probability, plus a Poisson(2) number
    of random impulse items."""
    basket: set[str] = set()
    for (bundle_items, _), prob in zip(BUNDLES, bundle_probs):
        if rng.random() < prob:
            for item in bundle_items:
                if rng.random() < 0.85:
                    basket.add(item)
    n_random = rng.poisson(2)
    if n_random > 0:
        random_items = rng.choice(PRODUCTS, size=int(min(n_random, 5)), replace=False)
        basket.update(random_items)
    return basket


def generate_transactions(
    n: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> list[set[str]]:
    """Generate ``n`` non-empty synthetic Singapore retail transactions.

    Each transaction is a set of product strings. Bundles fire with their
    listed probability; each item inside a firing bundle is kept with 0.85
    probability (random drop-out) so support is noisy. A Poisson number of
    random items is added on top to simulate impulse buys. Empty draws are
    re-drawn, so exactly ``n`` baskets are returned.
    """
    rng = np.random.default_rng(seed)
    base_probs = [prob for _, prob in BUNDLES]
    transactions: list[set[str]] = []
    while len(transactions) < n:
        basket = _draw_basket(rng, base_probs)
        if basket:
            transactions.append(basket)
    return transactions


# Habit model for two-trip shoppers (used by Exercise 5.4). A shopper is a
# "regular" for a bundle with probability HABIT_RATE_MULTIPLIER x the
# bundle's base rate; regulars fire that bundle on HABIT_FIRE_PROB of their
# trips, everyone else on CASUAL_FIRE_PROB of their trips.
HABIT_RATE_MULTIPLIER: float = 1.5
HABIT_FIRE_PROB: float = 0.6
CASUAL_FIRE_PROB: float = 0.03


def generate_shopper_trips(
    n_shoppers: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> tuple[list[set[str]], list[set[str]]]:
    """Two consecutive baskets per shopper: ``(this_trip, next_trip)``.

    Both trips are drawn independently from the SAME per-shopper habit
    profile, so this trip carries real but noisy evidence about the next
    one. The next trip is never used to build features — it only supplies
    a prediction target that is NOT a function of this trip's items.
    """
    rng = np.random.default_rng(seed)
    base = np.array([prob for _, prob in BUNDLES])
    this_trip: list[set[str]] = []
    next_trip: list[set[str]] = []
    while len(this_trip) < n_shoppers:
        habit = rng.random(len(BUNDLES)) < base * HABIT_RATE_MULTIPLIER
        probs = np.where(habit, HABIT_FIRE_PROB, CASUAL_FIRE_PROB)
        first = _draw_basket(rng, probs)
        second = _draw_basket(rng, probs)
        if first:
            this_trip.append(first)
            next_trip.append(second)
    return this_trip, next_trip


def transactions_to_onehot(transactions: list[set[str]]) -> pl.DataFrame:
    """One-row-per-transaction boolean matrix (columns = sorted PRODUCTS).

    Polars-native. mlxtend needs pandas, so callers convert with
    ``.to_pandas()`` at the mlxtend call site only.
    """
    all_items = sorted(PRODUCTS)
    rows = [{item: (item in txn) for item in all_items} for txn in transactions]
    return pl.DataFrame(rows)


def product_frequency(transactions: Iterable[set[str]]) -> dict[str, int]:
    """Count how many transactions contain each product."""
    counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            counts[item] += 1
    return dict(counts)


def print_transaction_summary(transactions: list[set[str]]) -> None:
    """One-liner summary + top 10 product frequency. Used by every file."""
    avg_basket = float(np.mean([len(t) for t in transactions]))
    print("=== Synthetic Singapore Retail Transactions ===")
    print(f"  Transactions: {len(transactions):,}")
    print(f"  Products:     {len(PRODUCTS)}")
    print(f"  Avg basket:   {avg_basket:.1f} items")

    freq = product_frequency(transactions)
    n = len(transactions)
    print("\n  Top 10 products by frequency:")
    for item, count in sorted(freq.items(), key=lambda kv: -kv[1])[:10]:
        print(f"    {item:<20} {count:>5} ({count / n:.1%})")


# ════════════════════════════════════════════════════════════════════════
# RULE HELPERS
# ════════════════════════════════════════════════════════════════════════


def format_itemset(items: Iterable[str]) -> str:
    """Deterministic pretty-print of a frozenset of items."""
    return ", ".join(sorted(items))


def categorise_rule(
    antecedent: frozenset[str],
    consequent: frozenset[str],
) -> tuple[set[str], set[str], str]:
    """Return (antecedent_categories, consequent_categories, relation_type)."""
    ant_cats = {CATEGORY_MAP.get(item, "other") for item in antecedent}
    con_cats = {CATEGORY_MAP.get(item, "other") for item in consequent}
    if ant_cats == con_cats:
        rel = "within-category complement"
    elif ant_cats & con_cats:
        rel = "cross-category with overlap"
    else:
        rel = "cross-category association"
    return ant_cats, con_cats, rel


def rules_to_polars(rules: list[dict]) -> pl.DataFrame:
    """Convert a list of rule dicts into a polars DataFrame for plotting."""
    return pl.DataFrame(
        {
            "antecedent": [format_itemset(r["antecedent"]) for r in rules],
            "consequent": [format_itemset(r["consequent"]) for r in rules],
            "support": [float(r["support"]) for r in rules],
            "confidence": [float(r["confidence"]) for r in rules],
            "lift": [float(r["lift"]) for r in rules],
        }
    )


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ML EXPERIMENT TRACKER — shared by every association-rules technique
# ════════════════════════════════════════════════════════════════════════
# Every M4 ex_5 lesson logs its mining + rule-evaluation + classifier runs
# into a single SQLite store so students can compare techniques side by side
# (Apriori vs FP-Growth runtime, rule-quality at varying thresholds,
# baseline vs rule-enhanced classifier metrics) after the lesson group ends.
# Mirrors m4_clustering_zoo / m4_dimreduction_zoo / m4_anomaly_zoo from
# ex_1 / ex_3 / ex_4.
#
# IMPORTANT: this store is SEPARATE from the other M4 stores. Per the
# SQLite contention trap (.session-notes), running two ExperimentTracker
# writers against the same SQLite file concurrently triggers `disk I/O
# error`. Each exercise group gets its own DB.

ASSOC_DB = "sqlite:///mlfp04_ex5_assoc_rules.db"
ASSOC_EXPERIMENT_NAME = "m4_assoc_rules_zoo"

_TRACKER_KEY_RE = re.compile(r"[^a-zA-Z0-9_.\-]")


def _slug(text: str) -> str:
    """Coerce free-form prose into a tracker-key-safe slug.

    ExperimentTracker enforces ``^[a-zA-Z_][a-zA-Z0-9_.\\-]*$`` on metric
    names. Replace forbidden characters with underscores; ensure a leading
    alpha character; collapse repeats.
    """
    s = _TRACKER_KEY_RE.sub("_", text.strip())
    s = re.sub(r"_+", "_", s).strip("_") or "metric"
    if not s[0].isalpha() and s[0] != "_":
        s = f"m_{s}"
    return s


def _finite(x: float) -> float:
    """Tracker rejects NaN/inf via MetricValueError; coerce to 0.0.

    Association-rule metrics (lift, conviction) can collapse to inf when
    confidence saturates at 1.0; aggregate scalars over empty filter sets
    can NaN. Every emit through this guard.
    """
    return float(x) if x == x and x not in (float("inf"), float("-inf")) else 0.0


async def _setup_engines_async() -> tuple[ExperimentTracker, str]:
    """Open the association-rules ExperimentTracker."""
    tracker = await ExperimentTracker.create(store_url=ASSOC_DB)
    return tracker, ASSOC_EXPERIMENT_NAME


def setup_engines() -> tuple[ExperimentTracker, str]:
    """Sync wrapper. Returns (tracker, experiment_name)."""
    return asyncio.run(_setup_engines_async())


def teardown_engines(tracker: ExperimentTracker) -> None:
    """Drain the aiosqlite worker threads before the script returns.

    kailash's AsyncSQLitePool spawns NON-DAEMON aiosqlite worker threads on
    first pool use. Python 3.13's ``Py_FinalizeEx`` joins non-daemon threads
    BEFORE running ``atexit`` handlers, so an atexit-based close runs too
    late — the interpreter hangs forever in ``wait_for_thread_shutdown``
    waiting on workers stuck in ``queue.get()``.

    Solutions MUST call ``teardown_engines(tracker)`` after the REFLECTION
    block (or use it via ``try: … finally: teardown_engines(tracker)``).
    See ``rules/patterns.md`` § "Async Resource Cleanup": real cleanup is
    the caller's responsibility; ``close()`` must NOT live in ``__del__``.
    """
    asyncio.run(tracker.close())


async def _track_run_async(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Log one lesson's run: scalar metrics + optional per-step series."""
    async with tracker.track(experiment=exp_name, run_name=run_name) as run:
        await run.log_params({k: str(v) for k, v in params.items()})
        for name, value in scalar_metrics.items():
            await run.log_metric(_slug(name), _finite(value))
        if series_metrics:
            for name, values in series_metrics.items():
                slug = _slug(name)
                for step, value in enumerate(values, start=1):
                    await run.log_metric(slug, _finite(value), step=step)


def track_run(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Sync wrapper for logging a single technique's run."""
    asyncio.run(
        _track_run_async(
            tracker, exp_name, run_name, params, scalar_metrics, series_metrics
        )
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 5.4: Rule-Based Features for Supervised Classification
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Engineer features from discovered association rules
#   - Compare a product-presence baseline against a rule-enhanced model
#   - Measure whether explicit rules add signal over raw one-hot features
#   - Build a prediction target that is NOT a function of the features
#   - Attribute model importance across product vs rule feature groups
#   - See the forward connection to matrix factorisation and neural nets
#
# PREREQUISITES:
#   - 01_apriori_from_scratch.py
#   - 03_rule_evaluation.py
#   - MLFP03 Exercise 1 (feature engineering)
#
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — rules as handcrafted co-occurrence features
#   2. Build — turn actionable rules into numeric feature columns
#   3. Train — predict next-trip breakfast-bundle purchase, baseline vs combined
#   4. Visualise — metric comparison + feature importance attribution
#   5. Apply — loyalty-programme next-trip offer targeting
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from collections import defaultdict
from itertools import combinations

import numpy as np
import polars as pl
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# ── Kailash-ML ExperimentTracker — every association-rules run logs here ─
tracker, exp_name = setup_engines()



## THEORY — Rules as Handcrafted Co-Occurrence Features


In [ ]:
# A linear model trained only on product-presence one-hot features sees
# each product as independent. It cannot represent "this basket contains
# the breakfast bundle" as a single signal — at best it learns a weight
# per product and adds them up.
#
# Association rules give you that missing structure for free. Every
# actionable rule X -> Y becomes one or two new binary features:
#
#   ant_present[r]   = 1 if the basket contains X
#   full_present[r]  = 1 if the basket contains X AND Y
#
# Plus a handful of aggregates:
#
#   n_rules_triggered  = how many rules fire on this basket
#   total_rule_lift    = sum of lifts of the rules that fire
#   max_rule_lift      = strongest rule that fires
#
# This is the MANUAL version of what matrix factorisation (Ex 7) and
# neural networks (Ex 8) will do AUTOMATICALLY. The progression is:
#
#   Manual rules  ->  Linear factorisation  ->  Nonlinear neural nets
#   explicit         compressed latent         learned non-linearity
#   interpretable    partially interpretable   least interpretable
#   cheap            medium                    expensive
#
# Ex 5.4 is the bottom rung. Every step above this file is a different
# way to discover co-occurrence structure without hand-writing rules.
#
# THE TARGET MUST NOT BE A FUNCTION OF THE FEATURES
# A tempting target is "big basket" (>= 6 items). But basket size is the
# row-sum of the one-hot product columns, so a linear model recovers it
# exactly (AUC = 1.0) and NO feature can add anything — a leak, not a
# finding. Here every shopper has TWO trips. Features come from THIS
# trip; the target is whether the NEXT trip contains the breakfast
# bundle (at least 2 of bread / butter / eggs). Both trips share the
# shopper's habits, so this trip is real but noisy evidence — exactly
# the setting where you can honestly ask whether rules add signal.



## MINI-APRIORI + RULE GENERATION (inline, so this file stands alone)


Inline Apriori — same contract as 01_apriori_from_scratch.apriori().


In [ ]:
def _apriori(
    transactions: list[set[str]], min_support: float
) -> dict[frozenset[str], float]:
    n = len(transactions)
    min_count = min_support * n
    item_counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            item_counts[item] += 1
    freq: dict[frozenset[str], float] = {}
    level: list[frozenset[str]] = []
    for item, count in item_counts.items():
        if count >= min_count:
            fs = frozenset([item])
            freq[fs] = count / n
            level.append(fs)
    k = 2
    while level:
        prev_set = set(level)
        candidates: set[frozenset[str]] = set()
        for i, a in enumerate(level):
            for b in level[i + 1 :]:
                u = a | b
                if len(u) == k and all((u - frozenset([it])) in prev_set for it in u):
                    candidates.add(u)
        if not candidates:
            break
        counts: dict[frozenset[str], int] = defaultdict(int)
        for txn in transactions:
            tf = frozenset(txn)
            for c in candidates:
                if c.issubset(tf):
                    counts[c] += 1
        level = []
        for c, ct in counts.items():
            if ct >= min_count:
                freq[c] = ct / n
                level.append(c)
        k += 1
    return freq


def _rules_from_itemsets(
    freq: dict[frozenset[str], float],
    min_confidence: float = 0.4,
    min_lift: float = 1.5,
) -> list[dict]:
    rules: list[dict] = []
    for itemset, support in freq.items():
        if len(itemset) < 2:
            continue
        items = list(itemset)
        for r in range(1, len(items)):
            for ant_tuple in combinations(items, r):
                antecedent = frozenset(ant_tuple)
                consequent = itemset - antecedent
                supp_ant = freq.get(antecedent)
                supp_con = freq.get(consequent)
                if supp_ant is None or supp_con is None:
                    continue
                confidence = support / supp_ant
                lift = confidence / supp_con
                if confidence >= min_confidence and lift > min_lift:
                    rules.append(
                        {
                            "antecedent": antecedent,
                            "consequent": consequent,
                            "support": support,
                            "confidence": confidence,
                            "lift": lift,
                        }
                    )
    rules.sort(key=lambda r: -r["lift"])
    return rules



## TASK 2 — BUILD: rule-based feature engineer


Columns produced:
      - ``rule{i}_antecedent`` — 1 if the antecedent is present
      - ``rule{i}_full``       — 1 if antecedent AND consequent are present
      - ``n_rules_triggered``  — count of fully-matched rules
      - ``total_rule_lift_x100`` — sum of matched rule lifts (scaled int)
      - ``max_rule_lift_x100``   — max matched rule lift (scaled int)


In [ ]:
def engineer_rule_features(
    transactions: list[set[str]],
    rules: list[dict],
) -> pl.DataFrame:
    rows: list[dict[str, int]] = []
    for txn in transactions:
        txn_set = frozenset(txn)
        row: dict[str, int] = {}
        total_lift = 0.0
        n_triggered = 0
        max_lift = 0.0

        for idx, rule in enumerate(rules):
            # TODO: compute ant_present (is the antecedent a subset of txn?)
            # and full_present (is antecedent UNION consequent a subset?).
            # Hint: frozenset.issubset(txn_set) returns a bool — wrap in int();
            # union two frozensets with the | operator.
            ant_present = ____
            full_present = ____
            row[f"rule{idx}_antecedent"] = ant_present
            row[f"rule{idx}_full"] = full_present
            if full_present:
                n_triggered += 1
                total_lift += float(rule["lift"])
                if float(rule["lift"]) > max_lift:
                    max_lift = float(rule["lift"])

        row["n_rules_triggered"] = n_triggered
        row["total_rule_lift_x100"] = int(total_lift * 100)
        row["max_rule_lift_x100"] = int(max_lift * 100)
        rows.append(row)

    return pl.DataFrame(rows).fill_null(0)



## TASK 3 — TRAIN: baseline vs combined classifiers


In [ ]:
# Features come from this trip; the label comes from the shopper's NEXT
# trip, which is never used to build a feature.
transactions, next_trips = generate_shopper_trips(n_shoppers=2500, seed=42)
print_transaction_summary(transactions)

TARGET_BUNDLE = frozenset({"bread", "butter", "eggs"})
# TODO: build the label from NEXT trips only — 1 if a shopper's next
# basket shares at least 2 items with TARGET_BUNDLE, else 0.
# Hint: set intersection (nxt & TARGET_BUNDLE) inside a list comprehension,
# wrapped in np.array([...]).
y = ____
print(
    f"\n  Target: next trip contains >= 2 of {sorted(TARGET_BUNDLE)} "
    "(breakfast bundle)"
)
print(f"  Positive rate: {y.mean():.1%}")

# --- Baseline features: raw product presence ---
onehot = transactions_to_onehot(transactions)
X_baseline = onehot.to_numpy().astype(np.float64)
print(f"\n  Baseline features (product presence): {X_baseline.shape[1]}")

# --- Mine rules and engineer features ---
print("\n=== Mining rules for feature engineering ===")
freq = _apriori(transactions, min_support=0.03)
rules = _rules_from_itemsets(freq, min_confidence=0.4, min_lift=1.5)
top_rules = rules[:20]
print(f"  Actionable rules found: {len(rules)}")
print(f"  Using top {len(top_rules)} rules as features")

rule_df = engineer_rule_features(transactions, top_rules)
X_rules = rule_df.to_numpy().astype(np.float64)
# TODO: stack baseline + rule features horizontally into X_combined.
# Hint: np.hstack([...]) — baseline columns first, rule columns second.
X_combined = ____
print(f"  Rule features:     {X_rules.shape[1]}")
print(f"  Combined features: {X_combined.shape[1]}")


def _split_and_scale(X: np.ndarray, y: np.ndarray, scale: bool):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    if scale:
        s = StandardScaler()
        X_tr = s.fit_transform(X_tr)
        X_te = s.transform(X_te)
    return X_tr, X_te, y_tr, y_te


results: dict[str, dict[str, float]] = {}

print("\n=== Model: Logistic Regression ===")
for name, X in [
    ("Baseline", X_baseline),
    ("Rules only", X_rules),
    ("Combined", X_combined),
]:
    X_tr, X_te, y_tr, y_te = _split_and_scale(X, y, scale=True)
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_tr, y_tr)
    y_pred = lr.predict(X_te)
    y_proba = lr.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_proba)
    results[f"LR: {name}"] = {"accuracy": acc, "f1": f1, "auc_roc": auc}
    print(f"  {name:<12} acc={acc:.4f} f1={f1:.4f} auc={auc:.4f}")

print("\n=== Model: Random Forest ===")
for name, X in [
    ("Baseline", X_baseline),
    ("Rules only", X_rules),
    ("Combined", X_combined),
]:
    X_tr, X_te, y_tr, y_te = _split_and_scale(X, y, scale=False)
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    )
    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_te)
    y_proba = rf.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_proba)
    results[f"RF: {name}"] = {"accuracy": acc, "f1": f1, "auc_roc": auc}
    print(f"  {name:<12} acc={acc:.4f} f1={f1:.4f} auc={auc:.4f}")



### Checkpoint


In [ ]:
assert (
    X_combined.shape[1] > X_baseline.shape[1]
), "Combined should add columns to the baseline, not replace them"
assert X_rules.shape[1] > 0, "Should have produced at least one rule feature"
lr_baseline = results["LR: Baseline"]["auc_roc"]
lr_combined = results["LR: Combined"]["auc_roc"]
rf_baseline = results["RF: Baseline"]["auc_roc"]
assert lr_baseline > 0.5, "Baseline LR should beat random"
assert rf_baseline > 0.5, "Baseline RF should beat random"
assert lr_baseline < 0.95, (
    "Baseline AUC is suspiciously close to 1.0 — check that the target is "
    "not computed from the feature columns (target leakage)"
)
assert (
    lr_combined >= lr_baseline - 0.05
), "Adding rule features should not significantly regress LR"
print("\n[ok] Checkpoint passed — rule-enhanced model trained + compared\n")



## TASK 4 — VISUALISE: feature importance attribution


In [ ]:
X_tr, X_te, y_tr, y_te = _split_and_scale(X_combined, y, scale=False)
rf_combined = RandomForestClassifier(
    n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
)
rf_combined.fit(X_tr, y_tr)
importances = rf_combined.feature_importances_

# TODO: split the RF importance vector into the product-feature sum and
# the rule-feature sum. The first X_baseline.shape[1] entries are
# product features; everything after is rule features.
# Hint: slice importances, then .sum() and float().
product_importance = ____
rule_importance = ____
total = product_importance + rule_importance

print("=== Feature Importance Attribution (RF combined) ===")
print(f"  Product features contribute: {product_importance / total:.1%}")
print(f"  Rule features contribute:    {rule_importance / total:.1%}")

all_feature_names = list(onehot.columns) + list(rule_df.columns)
top_idx = np.argsort(importances)[::-1][:15]
print("\n  Top 15 features in the combined model:")
for idx in top_idx:
    fname = all_feature_names[idx]
    ftype = "product" if idx < X_baseline.shape[1] else "rule"
    print(f"    [{ftype:>7}] {fname:<40} {importances[idx]:.4f}")

# Metric comparison frame (for notebooks to chart)
metric_rows = [{"model": k, **v} for k, v in results.items()]
metric_df = pl.DataFrame(metric_rows)
metric_df.write_csv(OUTPUT_DIR / "rule_features_metrics.csv")
print(f"\n  Saved: {OUTPUT_DIR / 'rule_features_metrics.csv'}")



### Visualisation


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# (A) Feature importance: product vs rule groups (pie + top-15 bar)
fig_imp = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "pie"}, {"type": "xy"}]],
    subplot_titles=["Importance by Group", "Top 15 Features"],
)
fig_imp.add_trace(
    go.Pie(
        labels=["Product Features", "Rule Features"],
        values=[product_importance, rule_importance],
        marker_colors=["#636EFA", "#EF553B"],
        textinfo="label+percent",
    ),
    row=1,
    col=1,
)
top_names = [all_feature_names[i] for i in top_idx]
top_vals = [float(importances[i]) for i in top_idx]
top_colors = ["#636EFA" if idx < X_baseline.shape[1] else "#EF553B" for idx in top_idx]
fig_imp.add_trace(
    go.Bar(
        x=top_vals,
        y=top_names,
        orientation="h",
        marker_color=top_colors,
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig_imp.update_layout(
    title="Feature Importance: Product (blue) vs Rule (red) Features",
    height=500,
    width=1000,
)
fig_imp.update_yaxes(autorange="reversed", row=1, col=2)
imp_path = OUTPUT_DIR / "04_feature_importance.html"
fig_imp.write_html(str(imp_path))
print(f"[viz] Feature importance: {imp_path}")

# (B) Accuracy improvement: baseline vs combined across models
model_names = list(results.keys())
acc_vals = [results[k]["accuracy"] for k in model_names]
auc_vals = [results[k]["auc_roc"] for k in model_names]
fig_acc = go.Figure()
fig_acc.add_trace(
    go.Bar(
        x=model_names,
        y=acc_vals,
        name="Accuracy",
        marker_color="#636EFA",
        text=[f"{v:.3f}" for v in acc_vals],
        textposition="outside",
    )
)
fig_acc.add_trace(
    go.Bar(
        x=model_names,
        y=auc_vals,
        name="AUC-ROC",
        marker_color="#EF553B",
        text=[f"{v:.3f}" for v in auc_vals],
        textposition="outside",
    )
)
fig_acc.update_layout(
    title="Model Comparison: Baseline vs Rules-Only vs Combined",
    xaxis_title="Model Variant",
    yaxis_title="Score",
    barmode="group",
    yaxis_range=[0, 1.1],
)
acc_path = OUTPUT_DIR / "04_accuracy_comparison.html"
fig_acc.write_html(str(acc_path))
print(f"[viz] Accuracy comparison: {acc_path}")

# INTERPRETATION — computed from this run, not assumed in advance.
lr_lift = results["LR: Combined"]["auc_roc"] - results["LR: Baseline"]["auc_roc"]
rf_lift = results["RF: Combined"]["auc_roc"] - results["RF: Baseline"]["auc_roc"]
print("\n=== Did rule features add signal? ===")
for model_name, lift in [("Logistic regression", lr_lift), ("Random forest", rf_lift)]:
    if lift > 0.01:
        verdict = "rules ADDED signal the product columns did not carry"
    elif lift < -0.01:
        verdict = "rules HURT — extra correlated columns added noise"
    else:
        verdict = "no measurable gain (within +/-0.01 AUC)"
    print(f"  {model_name:<20} AUC change {lift:+.3f} -> {verdict}")
print(
    "  Read this honestly. When each item is independent evidence of a\n"
    "  shopper's habit, a model that adds up per-product weights already\n"
    "  uses that evidence, so 'bread AND butter' columns add little AUC.\n"
    "  Rule features earn their place when the outcome depends on the\n"
    "  COMBINATION itself, and as named, auditable columns. Ex 7 (matrix\n"
    "  factorisation) and Ex 8 (neural nets) learn such structure without\n"
    "  pre-specified rules."
)



## TASK 5 — APPLY: loyalty-programme next-trip offer targeting


In [ ]:
# SCENARIO: A Singapore supermarket chain runs a grocery loyalty
# programme (assume ~1.5M members for illustration). The CRM team wants a
# model that scores each shopper right after checkout and predicts what
# they will buy on the NEXT trip, so the top 20% can receive a targeted
# next-trip offer (e.g. a breakfast-bundle voucher).
#
# Two constraints drive the design:
#   - The model must be AUDITABLE (regulator + internal governance)
#   - The feature engineering must be REPRODUCIBLE (monthly re-training)
#
# Rule-based features meet both. Every feature column has a business
# name you can trace to a specific association rule, and the mining
# pipeline reruns monthly against the previous 90 days of baskets. Your
# run above shows whether they also add accuracy on top of the product
# columns — if they do not, they can still be kept for explainability,
# but the decision should rest on the measured AUC change, not on hope.
#
# BUSINESS IMPACT (illustrative assumptions, not measured figures): if
# better targeting reaches an extra 1% of 1.5M members per month (15,000
# shoppers) and each activated offer contributes S$4 of margin, that is
# ~S$60K/month. Plug in YOUR measured AUC change and the programme's
# real offer economics before quoting a number. The rule-based column
# names also feed a "why did this shopper qualify?" explanation panel.
#
# LIMITATIONS:
#   - Rules captured here are support >= 3%; rarer categories (baby
#     care, pet food) need their own mining run with lower thresholds
#   - Two trips per shopper is a minimal history; production models
#     aggregate many past trips per shopper



## TRACK — Log baseline-vs-rule-enhanced metrics to ExperimentTracker


In [ ]:
# Logs all four model variants side by side (LR Baseline / LR Combined /
# RF Baseline / RF Combined) plus the product-vs-rule importance split.
# Series = top-15 RF importances so the M4 dashboard can visualise the
# combined-model ranking alongside the rule-quality distribution from
# lesson 03.

# TODO: AUC lift = combined AUC minus baseline AUC, for LR and for RF.
# Hint: results["LR: Combined"]["auc_roc"] holds one of the four AUCs.
lr_combined_lift = ____
rf_combined_lift = ____

track_run(
    tracker,
    exp_name,
    run_name=____,  # TODO: a stable name, e.g. "rule_features_vs_baseline"
    params={
        "algorithm": "rule_features",
        "implementation": "lr_plus_rf_baseline_vs_combined",
        "n_transactions": len(transactions),
        "n_baseline_features": int(X_baseline.shape[1]),
        "n_rule_features": int(X_rules.shape[1]),
        "n_combined_features": int(X_combined.shape[1]),
        "rf_n_estimators": 200,
        "rf_max_depth": 10,
    },
    scalar_metrics={
        "lr_baseline_auc": float(results["LR: Baseline"]["auc_roc"]),
        "lr_combined_auc": float(results["LR: Combined"]["auc_roc"]),
        "lr_auc_lift_combined_vs_baseline": float(lr_combined_lift),
        "rf_baseline_auc": float(results["RF: Baseline"]["auc_roc"]),
        "rf_combined_auc": float(results["RF: Combined"]["auc_roc"]),
        "rf_auc_lift_combined_vs_baseline": float(rf_combined_lift),
        "lr_baseline_f1": float(results["LR: Baseline"]["f1"]),
        "lr_combined_f1": float(results["LR: Combined"]["f1"]),
        "rf_baseline_f1": float(results["RF: Baseline"]["f1"]),
        "rf_combined_f1": float(results["RF: Combined"]["f1"]),
        "product_importance_share": float(product_importance / total),
        "rule_importance_share": float(rule_importance / total),
    },
    series_metrics={
        "rf_top15_importances": [float(importances[i]) for i in top_idx],
    },
)
print(f"  [tracked] Baseline-vs-combined classifier metrics logged to {exp_name}\n")



## DESTINATION-FIRST CLOSE — kailash-ml FeatureEngineer + TrainingPipeline


In [ ]:
# This lesson hand-rolled the rule-feature builder, the per-rule binary
# columns, the aggregate (n_rules_triggered / total_lift / max_lift) +
# four full sklearn classifier runs — ~340 lines to internalise rules-as-
# features end-to-end. The destination-first surface in Kailash is two
# engines stacked:
#
#   FeatureEngineer().generate(data, schema, strategies=["interactions"])
#       -> pairwise interaction columns (on 0/1 product columns, these
#          are exactly the co-occurrence features you hand-built)
#   TrainingPipeline(feature_store, registry).train(
#       data, schema, model_spec, eval_spec, experiment_name)
#       -> trained, evaluated, registered model
#
# In production you ship the rule-feature columns (kept verbatim for the
# auditability story above) PLUS the FeatureEngineer-generated set, then
# let TrainingPipeline train + rank LR vs RF vs gradient-boosted variants
# in one call. Manual rule features become one feature group among many,
# which is exactly the spectrum-of-discovery story this lesson opened on.

print("  Destination contract:")
print(
    "    FeatureEngineer().generate(data, schema, strategies=['interactions'])"
    "  -> co-occurrence cols"
)
print(
    "    TrainingPipeline(feature_store, registry).train("
    "data, schema, model_spec, eval_spec, experiment_name)  -> trained model"
)
print(
    f"  Today's run: hand-built {X_rules.shape[1]} rule features, "
    f"trained 4 sklearn variants — top AUC = {max(auc_vals):.4f}"
)
print()
print("  Manual rules are the bottom rung. The engine ships every rung above.\n")



## REFLECTION


[x] Engineered numeric features from discovered association rules
  [x] Compared baseline product-presence vs rule-enhanced models
  [x] Built a next-trip target that is not a function of the features,
      and guarded the baseline AUC against target leakage
  [x] Attributed feature importance across product and rule groups
  [x] Identified a production scenario (loyalty next-trip offers) where
      explicit rule features are valued for auditability
  [x] Pointed at the Kailash destination — FeatureEngineer +
      TrainingPipeline — that ships rule features alongside auto-discovered
      co-occurrence structure in one call

  KEY INSIGHT: Association rules are the MANUAL end of a spectrum that
  runs all the way to deep learning.

    Manual rules  ->  Linear factorisation  ->  Nonlinear neural nets
    (this file)      (Ex 7)                    (Ex 8)

  Every technique to the right of this one discovers the same
  co-occurrence structure, just with less human steering and less
  interpretability.

  Next: Exercise 6 moves to UNSTRUCTURED text — TF-IDF and BM25,
  NMF topic modelling, and topic quality evaluation via NPMI coherence.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

# Drain the aiosqlite worker threads so Py_Finalize doesn't hang.
teardown_engines(tracker)

